## Complete 2015 Dow 30 Factor Pipeline
This section consolidates all the data fetching, factor calculation, and point-in-time index membership logic into a clean, end-to-end pipeline.

In [30]:
import pandas as pd
import numpy as np
import yfinance as yf

In [31]:
# ------------------------------------------------------------
# 1) Dow 30 membership timeline
#    Changes are effective at the OPEN of the listed date.
# ------------------------------------------------------------
DOW_CHANGES = [
    ("2008-02-19", ["BAC", "CVX"], ["AA", "MO"]),
    ("2008-09-22", ["KFT"], ["AIG"]),
    ("2009-06-08", ["CSCO", "TRV"], ["C", "GM"]),
    ("2012-09-24", ["UNH"], ["KFT"]),
    ("2013-09-23", ["GS", "NKE", "V"], ["AA", "BAC", "HPQ"]),
    ("2015-03-19", ["AAPL"], ["T"]),
    ("2017-09-01", ["DWDP"], ["DD"]),
    ("2018-06-26", ["WBA"], ["GE"]),
    ("2019-04-02", ["DOW"], ["DWDP"]),
    ("2020-04-06", ["RTX"], ["UTX"]),
    ("2020-08-31", ["AMGN", "HON", "CRM"], ["XOM", "PFE", "RTX"]),
    ("2024-02-26", ["AMZN"], ["WBA"]),
    ("2024-11-08", ["NVDA", "SHW"], ["INTC", "DOW"]),
]

# ------------------------------------------------------------
# 2) Anchor basket: constituents active AFTER 2008-09-22 open
#    This avoids the earlier 2008 transition mess.
# ------------------------------------------------------------
ANCHOR_DATE = pd.Timestamp("2008-09-22")

DOW_AFTER_2008_09_22 = [
    "MMM", "AXP", "BAC", "BA", "CAT", "CVX", "C", "KO", "DD", "XOM",
    "GE", "HPQ", "HD", "INTC", "IBM", "JNJ", "JPM", "KFT", "MCD", "MRK",
    "MSFT", "PFE", "PG", "T", "UTX", "VZ", "WMT", "DIS", "TRV", "AA"
]

# ------------------------------------------------------------
# 3) Optional Yahoo fallback aliases
#    Use ONLY if the original symbol fails to download.
#    These are proxies, not true identity-equivalent renames.
# ------------------------------------------------------------
YAHOO_FALLBACK = {
    "UTX": "RTX",   # proxy only, not a true rename
    "KFT": "KHC",   # proxy only, not a true rename
    "DWDP": "DOW",  # proxy only, not a true rename

}


In [32]:
def get_dow_constituents_on(date):
    """
    Return the 30 Dow tickers active on a given date.
    """
    date = pd.Timestamp(date)
    basket = set(DOW_AFTER_2008_09_22)

    for change_date, added, removed in DOW_CHANGES:
        change_date = pd.Timestamp(change_date)
        if change_date > ANCHOR_DATE and change_date <= date:
            for r in removed:
                basket.discard(r)
            for a in added:
                basket.add(a)

    basket = sorted(basket)
    if len(basket) != 30:
        raise ValueError(f"{date.date()} has {len(basket)} names, expected 30.\n{basket}")
    return basket

def get_union_tickers(start_date, end_date):
    """
    Union of all official Dow symbols that appear at any point in range.
    """
    days = pd.date_range(start_date, end_date, freq="D")
    union = set()
    for d in days:
        union.update(get_dow_constituents_on(d))
    return sorted(union)

def try_download_symbol(symbol, start, end):
    """
    Try the official historical symbol first.
    If Yahoo returns no usable data, optionally try a fallback proxy.
    """
    candidates = [symbol]
    if symbol in YAHOO_FALLBACK:
        candidates.append(YAHOO_FALLBACK[symbol])

    for candidate in candidates:
        try:
            df = yf.download(
                candidate,
                start=start,
                end=end,
                auto_adjust=False,
                progress=False
            )
            if df is not None and not df.empty:
                return candidate, df
        except Exception:
            pass

    return None, pd.DataFrame()

def download_dow_range(start_date, end_date):
    """
    Download all needed symbols for a date range.
    Returns:
      - price_data: dict keyed by OFFICIAL symbol
      - pull_map: official symbol -> actual downloaded Yahoo symbol
      - membership: daily membership table
    """
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    official_symbols = get_union_tickers(start_date, end_date)

    price_data = {}
    pull_map = {}

    for sym in official_symbols:
        used_symbol, df = try_download_symbol(sym, start_date, end_date + pd.Timedelta(days=1))
        if df.empty:
            print(f"WARNING: no data found for {sym}")
        else:
            price_data[sym] = df
            pull_map[sym] = used_symbol

    membership_rows = []
    for d in pd.date_range(start_date, end_date, freq="D"):
        membership_rows.append({
            "date": d,
            "members": get_dow_constituents_on(d)
        })
    membership = pd.DataFrame(membership_rows)

    return price_data, pull_map, membership

# ------------------------------------------------------------
# Example: full year 2015
# ------------------------------------------------------------
'''
price_data_2015, pull_map_2015, membership_2015 = download_dow_range(
    "2015-01-01", "2015-12-31"
)

print("Pull map:")
for k, v in sorted(pull_map_2015.items()):
    print(f"{k} -> {v}")

print("\n2015-03-18 members:")
print(get_dow_constituents_on("2015-03-18"))

print("\n2015-03-19 members:")
print(get_dow_constituents_on("2015-03-19"))
'''

'\nprice_data_2015, pull_map_2015, membership_2015 = download_dow_range(\n    "2015-01-01", "2015-12-31"\n)\n\nprint("Pull map:")\nfor k, v in sorted(pull_map_2015.items()):\n    print(f"{k} -> {v}")\n\nprint("\n2015-03-18 members:")\nprint(get_dow_constituents_on("2015-03-18"))\n\nprint("\n2015-03-19 members:")\nprint(get_dow_constituents_on("2015-03-19"))\n'

In [33]:
REQUIRED_FIELDS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

def _flatten_columns(df):
    """
    Flatten Yahoo / pandas MultiIndex columns into simple strings.
    """
    if isinstance(df.columns, pd.MultiIndex):
        flat_cols = []
        for col in df.columns:
            parts = [str(x) for x in col if str(x) != "" and str(x).lower() != "nan"]
            flat_cols.append("_".join(parts))
        df.columns = flat_cols
    else:
        df.columns = [str(c) for c in df.columns]
    return df

def _standardize_single_symbol_df(df):
    """
    Standardize a single-symbol Yahoo dataframe to columns:
    date, Open, High, Low, Close, Adj Close, Volume
    """
    if df is None or df.empty:
        return pd.DataFrame()

    df = df.copy()
    df = _flatten_columns(df)

    # reset index so date becomes a column
    df = df.reset_index()

    # flatten again just in case reset_index creates odd labels
    df = _flatten_columns(df)

    # find the date column robustly
    date_candidates = [c for c in df.columns if c.lower() in ["date", "datetime", "index"]]
    if not date_candidates:
        # if nothing obvious exists, force first column to be date
        df = df.rename(columns={df.columns[0]: "date"})
    else:
        df = df.rename(columns={date_candidates[0]: "date"})

    # normalize common Yahoo column shapes
    rename_map = {}
    for c in df.columns:
        cl = c.lower()

        if cl == "open" or cl.startswith("open_"):
            rename_map[c] = "Open"
        elif cl == "high" or cl.startswith("high_"):
            rename_map[c] = "High"
        elif cl == "low" or cl.startswith("low_"):
            rename_map[c] = "Low"
        elif cl == "close" or cl.startswith("close_"):
            rename_map[c] = "Close"
        elif cl in ["adj close", "adj_close"] or cl.startswith("adj close_") or cl.startswith("adj_close_"):
            rename_map[c] = "Adj Close"
        elif cl == "volume" or cl.startswith("volume_"):
            rename_map[c] = "Volume"

    df = df.rename(columns=rename_map)

    # keep only date + required fields that exist
    keep_cols = ["date"] + [c for c in REQUIRED_FIELDS if c in df.columns]
    df = df[keep_cols].copy()

    # ensure date exists
    if "date" not in df.columns:
        raise ValueError(f"Could not identify date column. Columns were: {list(df.columns)}")

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df = df.sort_values("date").reset_index(drop=True)

    return df


In [34]:
def download_ohlcv_for_factor_inputs(start_date, end_date):
    """

    Returns:
      panel      : long-format daily panel
      pull_map   : dict of official_symbol -> yahoo_symbol_used
      membership : daily membership table
    """

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    official_symbols = get_union_tickers(start_date, end_date)

    price_parts = []
    pull_map = {}

    for sym in official_symbols:
        used_symbol, raw_df = try_download_symbol(
            sym,
            start_date.strftime("%Y-%m-%d"),
            (end_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        )

        if raw_df is None or raw_df.empty:
            print(f"WARNING: No data found for {sym}")
            continue

        try:
            df = _standardize_single_symbol_df(raw_df)
        except Exception as e:
            print(f"WARNING: Failed to standardize {sym} ({used_symbol}): {e}")
            continue

        if df.empty:
            print(f"WARNING: Standardized dataframe empty for {sym}")
            continue

        df["symbol"] = sym
        df["pull_symbol"] = used_symbol

        price_parts.append(df)
        pull_map[sym] = used_symbol

    if not price_parts:
        raise ValueError("No Yahoo Finance data was successfully downloaded.")

    panel = pd.concat(price_parts, ignore_index=True)

    # build daily membership table
    membership_rows = []
    for d in pd.date_range(start_date, end_date, freq="D"):
        membership_rows.append({
            "date": pd.Timestamp(d).normalize(),
            "members": get_dow_constituents_on(d)
        })

    membership = pd.DataFrame(membership_rows)

    # expand membership to date-symbol rows
    membership_expanded = []
    for _, row in membership.iterrows():
        for sym in row["members"]:
            membership_expanded.append({
                "date": row["date"],
                "symbol": sym,
                "in_dow": 1
            })

    membership_expanded = pd.DataFrame(membership_expanded)

    panel = panel.merge(
        membership_expanded,
        on=["date", "symbol"],
        how="left"
    )

    panel["in_dow"] = panel["in_dow"].fillna(0).astype(int)

    preferred_order = [
        "date", "symbol", "pull_symbol", "in_dow",
        "Open", "High", "Low", "Close", "Adj Close", "Volume"
    ]
    existing_order = [c for c in preferred_order if c in panel.columns]
    other_cols = [c for c in panel.columns if c not in existing_order]

    panel = panel[existing_order + other_cols].sort_values(
        ["date", "symbol"]
    ).reset_index(drop=True)

    return panel, pull_map, membership

# ============================================================
# DATA PULL MAP
# ============================================================
panel, pull_map, membership = download_ohlcv_for_factor_inputs(
    start_date="2007-01-01",
    end_date="2016-12-31"
)

print(panel.head())
print(panel.columns.tolist())

print("\nPull map:")
for k, v in sorted(pull_map.items()):
    print(f"{k} -> {v}")

panel.to_csv("dow_factor_inputs.csv", index=False)

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KFT']: YFPricesMissingError('possibly delisted; no price data found  (1d 2007-01-01 -> 2017-01-01)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['UTX']: YFTzMissingError('possibly delisted; no timezone found')


        date symbol pull_symbol  in_dow       Open       High        Low  \
0 2007-01-03     AA          AA       1  72.210152  72.234177  70.095512   
1 2007-01-03   AAPL        AAPL       0   3.081786   3.092143   2.925000   
2 2007-01-03    AXP         AXP       1  61.180000  61.900002  60.049999   
3 2007-01-03     BA          BA       1  88.900002  90.300003  88.449997   
4 2007-01-03    BAC         BAC       1  53.400002  54.180000  52.990002   

       Close  Adj Close      Volume  
0  70.479988  57.564415     3402538  
1   2.992857   2.510900  1238319600  
2  60.360001  45.023609     6142500  
3  89.169998  64.405701     4871700  
4  53.330002  36.031967    16028200  
['date', 'symbol', 'pull_symbol', 'in_dow', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

Pull map:
AA -> AA
AAPL -> AAPL
AXP -> AXP
BA -> BA
BAC -> BAC
C -> C
CAT -> CAT
CSCO -> CSCO
CVX -> CVX
DD -> DD
DIS -> DIS
GE -> GE
GS -> GS
HD -> HD
HPQ -> HPQ
IBM -> IBM
INTC -> INTC
JNJ -> JNJ
JPM -> JPM
KFT ->

# All data was pulled from 2008-end of 2016. change **output_start** and **output_end** in order to find the factor calculations of that year.

In [35]:
import numpy as np
import pandas as pd


# ============================================================
# FACTOR CALCULATION CODE
# Adds onto your existing `panel` dataframe
# ============================================================

def calculate_10_factors_from_panel(panel, output_start="2015-01-01", output_end="2015-12-31"):
    """
    Uses the already-downloaded panel to calculate:
      1. RSI (14-Day)
      2. Dist_MA20
      3. Bollinger Band Width
      4. Mom_12M_Refined
      5. MACD
      6. Dollar_Volume
      7. Volume_Shock
      8. MFI_14
      9. Historical_Volatility_20D
      10. Chaikin_Oscillator

    Parameters
    ----------
    panel : pd.DataFrame
        Must contain:
        date, symbol, pull_symbol, in_dow, High, Low, Close, Adj Close, Volume
    output_start : str or pd.Timestamp
        Start date for final returned factor dataframe
    output_end : str or pd.Timestamp
        End date for final returned factor dataframe

    Returns
    -------
    factors_output : pd.DataFrame
        Factor dataframe restricted to output window
    full_with_helpers : pd.DataFrame
        Full dataframe including helper columns over the entire downloaded range
    """

    df = panel.copy()

    # --------------------------------------------------------
    # 1) Basic cleaning
    # --------------------------------------------------------
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

    required_cols = ["date", "symbol", "High", "Low", "Close", "Adj Close", "Volume"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in panel: {missing}")

    numeric_cols = ["High", "Low", "Close", "Adj Close", "Volume"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    g = df.groupby("symbol", group_keys=False)

    # --------------------------------------------------------
    # 2) RSI (14-Day)
    # Formula:
    #   delta = P_t - P_{t-1}
    #   gain = max(delta, 0)
    #   loss = max(-delta, 0)
    #   avg_gain = Wilder smoothing over 14
    #   avg_loss = Wilder smoothing over 14
    #   RS = avg_gain / avg_loss
    #   RSI = 100 - 100/(1+RS)
    # --------------------------------------------------------
    def calc_rsi(series, window=14):
        delta = series.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)

        avg_gain = gain.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
        avg_loss = loss.ewm(alpha=1/window, adjust=False, min_periods=window).mean()

        rs = avg_gain / avg_loss.replace(0, np.nan)
        rsi = 100 - (100 / (1 + rs))
        return rsi

    df["RSI_14"] = g["Adj Close"].transform(lambda x: calc_rsi(x, 14))

    # --------------------------------------------------------
    # 3) Dist_MA20
    # Formula:
    #   MA20 = rolling mean of Adj Close over 20 days
    #   Dist_MA20 = (AdjClose / MA20) - 1
    # --------------------------------------------------------
    df["MA20"] = g["Adj Close"].transform(lambda x: x.rolling(20, min_periods=20).mean())
    df["Dist_MA20"] = (df["Adj Close"] / df["MA20"]) - 1

    # --------------------------------------------------------
    # 4) Bollinger Band Width
    # Formula:
    #   STD20 = rolling std of Adj Close over 20 days
    #   Upper = MA20 + 2*STD20
    #   Lower = MA20 - 2*STD20
    #   BB_Width = (Upper - Lower) / MA20
    # --------------------------------------------------------
    df["STD20"] = g["Adj Close"].transform(lambda x: x.rolling(20, min_periods=20).std())
    df["BB_Upper"] = df["MA20"] + 2 * df["STD20"]
    df["BB_Lower"] = df["MA20"] - 2 * df["STD20"]
    df["Bollinger_Band_Width"] = (df["BB_Upper"] - df["BB_Lower"]) / df["MA20"]

    # --------------------------------------------------------
    # 5) Z_score_20 (Distance from 20-day Moving Average in Std Devs)
    # Formula:
    #   ZScore_MA20 = (AdjClose - MA20) / STD20
    # --------------------------------------------------------
    df["ZScore_MA20"] = (df["Adj Close"] - df["MA20"]) / df["STD20"]

    # --------------------------------------------------------
    # 6) Mom_12M_Refined
    # Formula:
    #   skip most recent 1 month (~21 trading days)
    #   use prior 12-month momentum (~252 trading days)
    #   Mom_12M_Refined = (P_{t-21} / P_{t-252}) - 1
    # --------------------------------------------------------
    df["Mom_12M_Refined"] = g["Adj Close"].transform(lambda x: (x.shift(21) / x.shift(252)) - 1)

    # --------------------------------------------------------
    # 7) MACD
    # Formula:
    #   EMA12 = 12-day EMA of Adj Close
    #   EMA26 = 26-day EMA of Adj Close
    #   MACD = EMA12 - EMA26
    # --------------------------------------------------------
    ema12 = g["Adj Close"].transform(lambda x: x.ewm(span=12, adjust=False, min_periods=12).mean())
    ema26 = g["Adj Close"].transform(lambda x: x.ewm(span=26, adjust=False, min_periods=26).mean())
    df["MACD"] = ema12 - ema26

    # Optional extras if you want them for later
    df["MACD_Signal"] = g["MACD"].transform(lambda x: x.ewm(span=9, adjust=False, min_periods=9).mean())
    df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]

    # --------------------------------------------------------
    # 8) Dollar_Volume
    # Formula:
    #   Dollar_Volume = Adj Close * Volume
    # --------------------------------------------------------
    df["Dollar_Volume"] = df["Adj Close"] * df["Volume"]

    # --------------------------------------------------------
    # 9) Volume_Shock
    # Formula:
    #   AvgVol20 = rolling 20-day average volume
    #   Volume_Shock = (Volume / AvgVol20) - 1
    # --------------------------------------------------------
    df["AvgVol20"] = g["Volume"].transform(lambda x: x.rolling(20, min_periods=20).mean())
    df["Volume_Shock"] = (df["Volume"] / df["AvgVol20"]) - 1

    # --------------------------------------------------------
    # 10) Money Flow Index (MFI 14)
    # Formula:
    #   TypicalPrice = (High + Low + Close)/3
    #   RawMoneyFlow = TypicalPrice * Volume
    #   Positive if TP_t > TP_{t-1}
    #   Negative if TP_t < TP_{t-1}
    #   MoneyRatio = Sum(PosFlow,14) / Sum(NegFlow,14)
    #   MFI = 100 - 100/(1+MoneyRatio)
    # --------------------------------------------------------
    def calc_mfi(subdf, window=14):
        tp = (subdf["High"] + subdf["Low"] + subdf["Close"]) / 3.0
        rmf = tp * subdf["Volume"]

        tp_diff = tp.diff()
        pos_flow = np.where(tp_diff > 0, rmf, 0.0)
        neg_flow = np.where(tp_diff < 0, rmf, 0.0)

        pos_flow = pd.Series(pos_flow, index=subdf.index)
        neg_flow = pd.Series(neg_flow, index=subdf.index)

        pos_sum = pos_flow.rolling(window, min_periods=window).sum()
        neg_sum = neg_flow.rolling(window, min_periods=window).sum()

        money_ratio = pos_sum / neg_sum.replace(0, np.nan)
        mfi = 100 - (100 / (1 + money_ratio))
        return mfi

    df["MFI_14"] = (
        df.groupby("symbol", group_keys=False)
          .apply(lambda x: calc_mfi(x, 14))
          .reset_index(level=0, drop=True)
    )

    # --------------------------------------------------------
    # 11) Historical Volatility (20D, annualized)
    # Formula:
    #   LogRet_t = ln(P_t / P_{t-1})
    #   HistVol20 = std(LogRet,20) * sqrt(252)
    # --------------------------------------------------------
    df["LogRet"] = g["Adj Close"].transform(lambda x: np.log(x / x.shift(1)))
    df["Historical_Volatility_20D"] = g["LogRet"].transform(
        lambda x: x.rolling(20, min_periods=20).std() * np.sqrt(252)
    )

    # --------------------------------------------------------
    # 12) Chaikin Oscillator
    # Formula:
    #   MFM = ((Close-Low) - (High-Close)) / (High-Low)
    #   MFV = MFM * Volume
    #   ADL = cumulative sum of MFV
    #   Chaikin = EMA3(ADL) - EMA10(ADL)
    # --------------------------------------------------------
    def calc_chaikin(subdf):
        high = subdf["High"]
        low = subdf["Low"]
        close = subdf["Close"]
        vol = subdf["Volume"]

        hl_range = (high - low).replace(0, np.nan)
        mfm = ((close - low) - (high - close)) / hl_range
        mfv = mfm * vol
        adl = mfv.cumsum()

        ema3 = adl.ewm(span=3, adjust=False, min_periods=3).mean()
        ema10 = adl.ewm(span=10, adjust=False, min_periods=10).mean()

        return ema3 - ema10

    df["Chaikin_Oscillator"] = (
        df.groupby("symbol", group_keys=False)
          .apply(calc_chaikin)
          .reset_index(level=0, drop=True)
    )

    # --------------------------------------------------------
    # 13) Final factor dataframe
    # --------------------------------------------------------
    factor_cols = [
        "date", "symbol", "pull_symbol", "in_dow",
        "RSI_14",
        "Dist_MA20",
        "Bollinger_Band_Width",
        "ZScore_MA20",
        "Mom_12M_Refined",
        "MACD",
        "Dollar_Volume",
        "Volume_Shock",
        "MFI_14",
        "Historical_Volatility_20D",
        "Chaikin_Oscillator"
    ]

    factors_full = df[factor_cols].copy()

    output_start = pd.Timestamp(output_start)
    output_end = pd.Timestamp(output_end)

    factors_output = factors_full[
        (factors_full["date"] >= output_start) &
        (factors_full["date"] <= output_end)
    ].copy()

    return factors_output, df


# ============================================================
# RUN FACTOR CALCULATION ON YOUR EXISTING `panel`
# ============================================================
panel, pull_map, membership = download_ohlcv_for_factor_inputs(
    start_date="2007-01-01",
    end_date="2016-12-31"
)

factors_2015, full_debug_df = calculate_10_factors_from_panel(
    panel,
    output_start="2015-01-01",
    output_end="2015-12-31"
)

print(factors_2015.head())
print(factors_2015.columns.tolist())

# Save outputs
# factors_2015.to_csv("dow_10_factors_2015.csv", index=False)
# full_debug_df.to_csv("dow_10_factors_full_debug.csv", index=False)

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KFT']: YFPricesMissingError('possibly delisted; no price data found  (1d 2007-01-01 -> 2017-01-01)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['UTX']: YFTzMissingError('possibly delisted; no timezone found')


           date symbol pull_symbol  in_dow     RSI_14  Dist_MA20  \
2014 2015-01-02     AA          AA       0  48.376668   0.009183   
2015 2015-01-05     AA          AA       0  35.960564  -0.042346   
2016 2015-01-06     AA          AA       0  38.009179  -0.029464   
2017 2015-01-07     AA          AA       0  44.756903  -0.001679   
2018 2015-01-08     AA          AA       0  51.209481   0.027198   

      Bollinger_Band_Width  ZScore_MA20  Mom_12M_Refined      MACD  \
2014              0.166482     0.220644         0.656979 -0.376620   
2015              0.146594    -1.155452         0.642121 -0.496391   
2016              0.126615    -0.930817         0.651233 -0.564947   
2017              0.118199    -0.056808         0.621916 -0.542642   
2018              0.116206     0.936214         0.517022 -0.440484   

      Dollar_Volume  Volume_Shock     MFI_14  Historical_Volatility_20D  \
2014   1.541480e+08     -0.383447  56.933281                   0.350789   
2015   3.019994e+08 

/tmp/ipykernel_18547/3060235253.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: calc_mfi(x, 14))
/tmp/ipykernel_18547/3060235253.py:226: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_chaikin)


In [36]:
# Keep only rows where the stock is actually in the Dow that day
factors_2015_in_dow = factors_2015[factors_2015["in_dow"].astype(int) == 1].copy()
factors_2015_in_dow.sort_values(by=['date', "symbol"], inplace=True)

# factors_2015_in_dow.to_csv("dow_10_factors_in_dow_2015.csv", index=False)

print(factors_2015_in_dow.head(10))

            date symbol pull_symbol  in_dow     RSI_14  Dist_MA20  \
7050  2015-01-02    AXP         AXP       1  53.514801   0.003733   
9568  2015-01-02     BA          BA       1  53.943203   0.014953   
17122 2015-01-02    CAT         CAT       1  38.718993  -0.012616   
19640 2015-01-02   CSCO        CSCO       1  57.369481   0.007789   
22158 2015-01-02    CVX         CVX       1  53.339116   0.031170   
24676 2015-01-02     DD          DD       1  44.106698  -0.003559   
27194 2015-01-02    DIS         DIS       1  57.566415   0.009590   
29712 2015-01-02     GE          GE       1  41.942660  -0.009258   
32230 2015-01-02     GS          GS       1  56.063050   0.008492   
34748 2015-01-02     HD          HD       1  60.794498   0.019984   

       Bollinger_Band_Width  ZScore_MA20  Mom_12M_Refined      MACD  \
7050               0.069325     0.215419         0.051009  0.632021   
9568               0.114817     0.520929        -0.009527  0.615906   
17122              0.116677

# USE THESE CSV FILES FOR THE ANALYSIS. THESE ARE RAW NUMBERS

In [37]:
import os

# 1. Define the output folder name
output_folder = "dow_factors_data"

# 2. Create the folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created directory: {output_folder}")

# ============================================================
# LOOP OVER YEARS AND EXPORT FACTORS
# ============================================================

for year in range(2009, 2017):  # 2009 → 2016 inclusive
    start = f"{year}-01-01"
    end   = f"{year}-12-31"

    print(f"Processing {year}...")

    factors_year, _ = calculate_10_factors_from_panel(
        panel,
        output_start=start,
        output_end=end
    )

    # Filter to only Dow members
    factors_year = factors_year[factors_year["in_dow"] == 1].copy()

    # Sort properly
    factors_year = factors_year.sort_values(by=["date", "symbol"])

    # 3. Construct the filename with the folder path
    filename = f"dow_10_factors_{year}.csv"
    file_path = os.path.join(output_folder, filename)

    # Export to the specific path
    factors_year.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")

Created directory: dow_factors_data
Processing 2009...


/tmp/ipykernel_18547/3060235253.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: calc_mfi(x, 14))
/tmp/ipykernel_18547/3060235253.py:226: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_chaikin)


Saved: dow_factors_data/dow_10_factors_2009.csv
Processing 2010...


/tmp/ipykernel_18547/3060235253.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: calc_mfi(x, 14))
/tmp/ipykernel_18547/3060235253.py:226: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_chaikin)


Saved: dow_factors_data/dow_10_factors_2010.csv
Processing 2011...


...

Saved: dow_factors_data/dow_10_factors_2014.csv
Processing 2015...


/tmp/ipykernel_18547/3060235253.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: calc_mfi(x, 14))
/tmp/ipykernel_18547/3060235253.py:226: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_chaikin)


Saved: dow_factors_data/dow_10_factors_2015.csv
Processing 2016...


/tmp/ipykernel_18547/3060235253.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: calc_mfi(x, 14))
/tmp/ipykernel_18547/3060235253.py:226: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_chaikin)


Saved: dow_factors_data/dow_10_factors_2016.csv


In [39]:
panel, pull_map, membership = download_ohlcv_for_factor_inputs(
    start_date="2018-01-01",
    end_date="2025-12-31"
)

factors_2020_to_2025, full_debug_df = calculate_10_factors_from_panel(
    panel,
    output_start="2020-01-01",
    output_end="2025-12-31"
)


factors_2020_to_2025 = factors_2020_to_2025[factors_2020_to_2025["in_dow"] == 1].copy()

# Sort properly
factors_2020_to_2025 = factors_2020_to_2025.sort_values(by=["date", "symbol"])

print(factors_2020_to_2025.head())
print(factors_2020_to_2025.columns.tolist())

#Save outputs
factors_2020_to_2025.to_csv("dow_10_factors_2020_to_2025.csv", index=False)
full_debug_df.to_csv("dow_10_factors_full_debug.csv", index=False)

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['DWDP']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['UTX']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')


/tmp/ipykernel_18547/3060235253.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: calc_mfi(x, 14))
/tmp/ipykernel_18547/3060235253.py:226: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_chaikin)


            date symbol pull_symbol  in_dow     RSI_14  Dist_MA20  \
503   2020-01-02   AAPL        AAPL       1  84.607663   0.075794   
6536  2020-01-02    AXP         AXP       1  64.559847   0.024807   
8547  2020-01-02     BA          BA       1  43.516499  -0.012085   
10558 2020-01-02    CAT         CAT       1  65.677930   0.032601   
14580 2020-01-02   CSCO        CSCO       1  68.330907   0.054939   

       Bollinger_Band_Width  ZScore_MA20  Mom_12M_Refined      MACD  \
503                0.149261     2.031194         0.697878  2.108647   
6536               0.081405     1.218932         0.243515  1.361286   
8547               0.115431    -0.418784         0.122151 -7.198674   
10558              0.075269     1.732517         0.163255  1.533004   
14580              0.153973     1.427250         0.070149  0.569121   

       Dollar_Volume  Volume_Shock     MFI_14  Historical_Volatility_20D  \
503     9.808849e+09      0.170276  79.698711                   0.146879   
6536  